In [1]:
import json
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/seminar/

/content/drive/MyDrive/seminar


In [5]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import os
from IPython.display import display

# Model ID
model_id = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Optional but important: set patch size if missing
if hasattr(processor.image_processor, "patch_size") and processor.image_processor.patch_size is None:
    processor.image_processor.patch_size = 14
else:
    processor.patch_size = 14

# Load image
#image_path = "/content/drive/MyDrive/seminar/0cbaca12e05803e6301f8f4a92b47565.png"
images_path ="/content/drive/MyDrive/seminar/nova_brain/images"


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [6]:
results = []

for image_name in os.listdir(images_path):
  print(image_name)
  image_path = os.path.join(images_path, image_name)
  image = Image.open(image_path).convert("RGB")

  # visualize the image
  #display(image)

  #prompt = """USER: <image>\n You are a brain MRI medical assistant. Carefully analyze and describe the image. The description should be in the form of a caption with the most important findings. ASSISTANT:"""

  prompt = (
    "USER: <image>\n"
    "You are a medical assistant specialized in brain MRI interpretation. Carefully analyze the image and provide a concise caption summarizing the most important findings.\n"
    "ASSISTANT:"
  )

  # Prepare inputs (text prompt + image)
  inputs = processor(
      text=prompt,
      images=image,
      return_tensors="pt"
  )

  #print("inputs:", inputs)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  # Run inference
  with torch.no_grad():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=128,
          min_new_tokens=1,
          do_sample=True,
          temperature=0.7,
          pad_token_id=processor.tokenizer.pad_token_id,
          eos_token_id=processor.tokenizer.eos_token_id,
          use_cache=True
      )

  # Decode result
  full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

  # Extract only the assistant's part
  if "ASSISTANT:" in full_response:
      assistant_response = full_response.split("ASSISTANT:")[-1].strip()
  else:
      assistant_response = full_response

  #print("Full response:", full_response)
  print("Assistant response:", assistant_response)

  results.append({
            'image': image_name,
            'prediction': assistant_response
        })

case0005_002.png
Assistant response: The axial T2-weighted MRI image reveals bilateral basal ganglia hyperintensities and a right-sided frontal lobe hyperintensity. These hyperintensities indicate areas of increased signal intensity, which may be associated with various underlying conditions or abnormalities. Further evaluation and clinical correlation are needed to determine the cause and significance of these findings.
case0070_008.png
Assistant response: The brain MRI shows mild atrophy and mild widening of the lateral ventricles. Additionally, there is minimal cortical atrophy in the right parietal, temporal, and occipital regions. These findings may indicate changes in the brain's structure and function, but further evaluation and clinical correlation are needed to determine the cause and significance of these findings.
case0127_002.png
Assistant response: The MRI image shows several findings, including generalized atrophy, enlargement of the lateral ventricles, an enlarged Sylvia

In [7]:
import csv

#save all results to a csv file with 2 columns - image and prediction
csv_output_path = "/content/drive/MyDrive/seminar/mri_descriptions_v2.csv"

with open(csv_output_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'prediction']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header
    writer.writeheader()

    # Write data
    for result in results:
        writer.writerow(result)

In [ ]:
results = []

for image_name in os.listdir(images_path):
    image_path = os.path.join(images_path, image_name)
    image = Image.open(image_path).convert("RGB")
    prompt = f"""USER: <image>\n Carefully analyze the provided brain MRI and locate the abnormal region. Provide ONLY the bounding box coordinates for the abnormal region in the following format: (x1, y1, x2, y2). ASSISTANT:"""
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )

    #print("inputs:", inputs)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Run inference
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50, # Increased max_new_tokens to allow for full coordinate output
            min_new_tokens=1,
            do_sample=True,
            temperature=0.7,
            top_p = 0.8,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True
        )

    # Decode result
    full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Extract only the assistant's part
    if "ASSISTANT:" in full_response:
        assistant_response = full_response.split("ASSISTANT:")[-1].strip()
    else:
        assistant_response = full_response

    #print("Full response:", full_response)
    print("Assistant response:", assistant_response)

    results.append({
              'image': image_name,
              'prediction': assistant_response
          })

Assistant response: The abnormal region is located in the right frontal lobe. The bounding box coordinates for this region are (31, 17, 42, 30).
Assistant response: The abnormal region is located at (15, 25, 22, 33).
Assistant response: The abnormal region in the brain MRI is located at (34, 21, 40, 30).
Assistant response: The abnormal region in the brain MRI is located at (35, 27, 38, 26).
Assistant response: The abnormal region is located in the left cerebellar hemisphere. The coordinates for this region are (21, 20, 28, 26).
Assistant response: The abnormal region in the brain MRI is located at (55, 26, 60, 32).
Assistant response: The abnormal region is located at (47, 10, 57, 40).
Assistant response: The abnormal region is located in the right temporal lobe. The bounding box coordinates for this region are (20, 20, 40, 40).
Assistant response: The abnormal region is located at (15, 12, 16, 11).
Assistant response: The abnormal region is located in the left occipital lobe. The bou

In [ ]:
import csv
csv_output_path = "/content/drive/MyDrive/seminar/mri_predictions_grounding.csv"

with open(csv_output_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'prediction']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header
    writer.writeheader()

    # Write data
    for result in results:
        writer.writerow(result)

In [8]:
#load the annotations file
import json
annotations_path = "/home/useradd/seminar/Mecial-Imaging-Seminar/src/data/nova_brain/annotations.json"
with open(annotations_path, 'r') as f:
    annotations = json.load(f)

cases = list(annotations.keys())
for case in cases:
    imgs = list(annotations[case]['image_findings'].keys())
    print(imgs)



['case0061_001.png', 'case0061_002.png']
['case0246_001.png']
['case0115_007.png', 'case0115_001.png', 'case0115_002.png', 'case0115_003.png', 'case0115_004.png', 'case0115_005.png', 'case0115_006.png']
['case0088_004.png', 'case0088_001.png', 'case0088_002.png', 'case0088_003.png']
['case0034_003.png', 'case0034_004.png', 'case0034_001.png', 'case0034_002.png', 'case0034_005.png']
['case0005_003.png', 'case0005_001.png', 'case0005_002.png']
['case0080_006.png', 'case0080_002.png', 'case0080_003.png', 'case0080_004.png', 'case0080_005.png', 'case0080_001.png']
['case0150_001.png', 'case0150_002.png', 'case0150_003.png', 'case0150_004.png']
['case0127_001.png', 'case0127_002.png', 'case0127_003.png']
['case0275_001.png', 'case0275_002.png']
['case0274_001.png', 'case0274_002.png']
['case0043_001.png', 'case0043_002.png', 'case0043_005.png', 'case0043_003.png', 'case0043_004.png']
['case0212_001.png', 'case0212_002.png', 'case0212_003.png', 'case0212_004.png']
['case0113_001.png', 'case0